# 비전 모델 1 학습 (Colab)

종류 분류는 공개 데이터(DeepFashion2)로 먼저 학습하고, 촬영한 사진으로 이어서 학습한다. 방향 확인은 촬영한 사진으로 학습한다.

**실행 전 준비**
1. 런타임 → 런타임 유형 변경 → GPU 선택
2. 구글 드라이브에 `vision_model1` 폴더를 올린다. (`vision/`, `requirements.txt`, 이 노트북)
3. `python -m vision.deepfashion2 --src <압축을 푼 폴더> --zip` 으로 만든 `public.zip` 을 드라이브의 `vision_model1/data/` 에 올린다.
4. 촬영 원본이 있으면 `vision_model1/data/raw/` 에 넣는다. 폴더 규칙은 `README.md` 의 학습 항목 참고. 아직 없으면 2번, 3번 항목은 자동으로 건너뛴다.

학습된 가중치는 드라이브의 `vision_model1/weights/` 에 남는다.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

`PROJECT` 는 드라이브에 올린 `vision_model1` 폴더의 경로다. 올린 위치가 다르면 고친다.

In [ ]:
PROJECT = '/content/drive/MyDrive/vision_model1'
%cd $PROJECT
!ls

In [ ]:
!pip install -q -r requirements.txt

## 1. 공개 데이터로 종류 분류 학습

`public.zip` 을 Colab의 로컬 디스크에 푼다. 드라이브에서 한 장씩 읽는 것보다 학습이 훨씬 빠르다.

In [ ]:
!mkdir -p /content/public && unzip -q -o data/public.zip -d /content/public
!for d in /content/public/*; do echo "$(basename $d): $(ls $d | wc -l)장"; done

무작위로 몇 장 뽑아 옷이 제대로 잘렸는지 확인한다.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
from vision.dataset import scan_flat

samples = scan_flat('/content/public')
picked = random.sample(samples, min(12, len(samples)))

fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
for ax in axes.flat:
    ax.axis('off')
for ax, s in zip(axes.flat, picked):
    ax.imshow(Image.open(s.path))
    ax.set_title(s.category)
plt.show()

촬영한 사진이 아직 없으면 공개 데이터의 일부를 떼어 검증에 쓴다. 이때의 검증 정확도는 실제 카메라 환경의 성능이 아니라, 학습이 제대로 되고 있는지 보는 용도다.

In [ ]:
!python -m vision.train --task category --extra /content/public --epochs 15 --batch-size 64

In [ ]:
import json

def plot_history(tasks):
    fig, axes = plt.subplots(1, len(tasks), figsize=(5.5 * len(tasks), 3.5), squeeze=False)
    for ax, task in zip(axes[0], tasks):
        history = json.load(open(f'weights/{task}_history.json'))
        epochs = [h['epoch'] for h in history]
        ax.plot(epochs, [h['train_acc'] for h in history], label='train')
        ax.plot(epochs, [h['val_acc'] for h in history], label='val')
        ax.set_title(task)
        ax.set_xlabel('epoch')
        ax.set_ylabel('accuracy')
        ax.set_ylim(0, 1.05)
        ax.legend()
    plt.show()

plot_history(['category'])

## 2. 촬영한 사진으로 이어서 학습

촬영 원본이 `data/raw` 에 있을 때만 실행한다. 아직 없으면 아래 칸은 모두 건너뛴다.

In [ ]:
from pathlib import Path

raw = Path('data/raw')
HAS_PHOTOS = raw.is_dir() and any(p.is_dir() for p in raw.iterdir())
print('촬영 원본을 찾았습니다.' if HAS_PHOTOS else '촬영 원본이 없어 2번, 3번 항목은 건너뜁니다.')

`data/raw` 의 원본에서 옷을 찾아 잘라 `data/crops` 에 저장한다. 처음 실행하면 탐지 모델을 내려받는다.

In [ ]:
if HAS_PHOTOS:
    !python -m vision.pipeline crop

In [ ]:
if HAS_PHOTOS:
    from vision.dataset import scan

    samples = scan('data/crops')
    picked = random.sample(samples, min(12, len(samples)))

    fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
    for ax in axes.flat:
        ax.axis('off')
    for ax, s in zip(axes.flat, picked):
        ax.imshow(Image.open(s.path))
        ax.set_title(f'{s.category} / {s.view}')
    plt.show()

1번에서 만든 가중치에서 이어서 학습한다. 공개 데이터만으로 학습한 가중치는 비교할 수 있도록 `category_public.pt` 로 남겨 둔다.
학습과 검증은 옷 단위로 나누고, 검증 정확도가 가장 높은 에폭을 저장한다.

In [ ]:
if HAS_PHOTOS:
    !cp weights/category.pt weights/category_public.pt
    !python -m vision.train --task category --init weights/category.pt --lr 1e-4 --epochs 20

In [ ]:
if HAS_PHOTOS:
    !python -m vision.train --task view --epochs 20

In [ ]:
if HAS_PHOTOS:
    plot_history(['category', 'view'])

## 3. 전체 실행 확인

옷 한 벌의 원본 사진을 넣어 탐지 → 크롭 → 분류가 이어지는지 본다. `garment` 는 `data/raw` 안의 옷 폴더 이름이다.

In [ ]:
if HAS_PHOTOS:
    from vision.pipeline import Pipeline, merge_category

    garment = sorted(p.name for p in Path('data/raw').iterdir() if p.is_dir())[0]
    view_dirs = sorted(d for d in (Path('data/raw') / garment).iterdir() if d.is_dir())
    photos = [sorted(d.iterdir())[0] for d in view_dirs]

    pipe = Pipeline()
    results = pipe.run_garment(photos)

    fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4.5), squeeze=False)
    for ax, photo, r in zip(axes[0], photos, results):
        ax.imshow(r.image)
        ax.set_title(f'{photo.parent.name} -> {r.view} / {r.category}\nready={r.ready}')
        ax.axis('off')
    plt.show()
    print(garment, '->', merge_category(results))